# 11 Split-Half Ceiling

How well do two groups of people agree with each other? This is the ceiling for any model that is scored against the human norms.

For each cue, the people who answered it are split at random into two halves. The score is the **top-10 feature-mass coverage** used in the proposal: take the top 10 features of half B (most people first), and ask how much of their mass (the percent of half B's people who listed them) is recovered by the features that half A produced at all. It is scored both ways (A against B, B against A) and averaged, over 100 random splits per cue.

Only cues with 20 or more people (`n_final`) are scored, so that each half has at least 10 people.

Output: `processed/11_split_half_ceiling.csv` (one row per cue) and `processed/11_split_half_summary.csv`.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
N_SPLITS = 100
MIN_PEOPLE = 20

In [2]:
words = pd.read_csv("processed/08_words.csv", usecols=["subject_id", "cue", "word", "n_final"],
                    dtype={"subject_id": str}, keep_default_na=False)
print(f"{len(words):,} rows, {words['cue'].nunique():,} cues")

952,792 rows, 3,722 cues


In [3]:
def coverage(binary, half_a, half_b):
    """Share of half B's top-10 feature mass that half A's features cover."""
    in_a = binary[half_a].any(axis=0)
    share_b = binary[half_b].mean(axis=0) * 100
    top = np.argsort(-share_b + rng.random(share_b.size) * 1e-6)[:10]    # random tie-break
    top = top[share_b[top] > 0]
    return share_b[top][in_a[top]].sum() / share_b[top].sum() * 100

rows = []
for cue, g in words.groupby("cue"):
    people = g["subject_id"].unique()
    if len(people) < MIN_PEOPLE:
        continue
    pid = {p: i for i, p in enumerate(people)}
    feats = {f: j for j, f in enumerate(g["word"].unique())}
    binary = np.zeros((len(people), len(feats)), dtype=bool)
    binary[g["subject_id"].map(pid).values, g["word"].map(feats).values] = True
    scores = []
    for _ in range(N_SPLITS):
        order = rng.permutation(len(people))
        a, b = order[: len(people) // 2], order[len(people) // 2 :]
        scores.append((coverage(binary, a, b) + coverage(binary, b, a)) / 2)
    rows.append((cue, len(people), len(feats), np.mean(scores), np.std(scores)))

ceiling = pd.DataFrame(rows, columns=["cue", "n_people", "n_features", "split_half_coverage", "sd_over_splits"])
ceiling.round(3).to_csv("processed/11_split_half_ceiling.csv", index=False)
print(f"{len(ceiling):,} cues scored")

3,720 cues scored


In [4]:
summary = ceiling["split_half_coverage"].describe().round(1).to_frame("split-half coverage")
summary.loc["cues scored"] = len(ceiling)
summary.loc["cues skipped (fewer than 20 people)"] = words["cue"].nunique() - len(ceiling)
summary.to_csv("processed/11_split_half_summary.csv")
summary

,split-half coverage
count,3720.0
mean,86.9
std,10.8
min,37.5
25%,80.4
50%,88.8
75%,95.9
max,100.0
cues scored,3720.0
cues skipped (fewer than 20 people),2.0


In [5]:
# does it depend on how many people answered?
ceiling.groupby(pd.cut(ceiling["n_people"], [19, 29, 39, 69, 400]), observed=True)["split_half_coverage"].agg(["size", "mean"]).round(1)

,size,mean
n_people,,
"(19, 29]",781,78.1
"(29, 39]",1135,80.8
"(39, 69]",1223,93.1
"(69, 400]",581,97.4
